# 🧠 Klasifikasi Citra MRI Tumor Otak dengan CNN
### Transfer Learning menggunakan **MobileNetV2**

Notebook ini membangun model *Convolutional Neural Network* (CNN) untuk mengklasifikasikan citra MRI otak ke dalam beberapa kelas (glioma, meningioma, notumor, pituitary).

**Pemetaan kriteria tugas ke bagian notebook**

| No | Kriteria | Bagian |
|----|----------|--------|
| 1 | Contoh data dari setiap kelas | 3 |
| 2 | Grafik jumlah gambar per kelas | 4 |
| 3 | Data Augmentation + contoh hasilnya (opsional) | 6 |
| 4 | Transfer Learning **MobileNetV2** | 8 |
| 5 | Callbacks (berhenti saat tidak ada perkembangan) | 9 |
| 6 | Grafik histori akurasi & loss | 12 |
| 7 | Confusion matrix & classification report | 13 |

**Alur kerja:** muat data → eksplorasi → augmentasi → bangun model → latih (2 fase) → evaluasi.
Sumber data: https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset/data?select=Training
Catatan: Unduh data sendiri

## 1. Import Library & Konfigurasi

- mengimpor seluruh library yang dibutuhkan (**TensorFlow/Keras** untuk model, **scikit-learn** untuk evaluasi, **matplotlib/seaborn** untuk visualisasi),
- menetapkan **random seed** agar hasil percobaan dapat direproduksi,
- mengumpulkan semua **hyperparameter** di satu tempat sehingga mudah diubah.

Beberapa pilihan penting:
- `IMG_SIZE = (224, 224)` → ukuran input standar MobileNetV2 saat dilatih pada ImageNet.
- `VAL_SPLIT = 0.2` → 20% data *Training* dipisah menjadi data validasi (dipakai callbacks untuk memantau performa).
- Pelatihan dibagi dua fase (`EPOCHS_HEAD` dan `EPOCHS_FINE`), dijelaskan pada bagian 10 dan 11.

In [ ]:
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import MaxNLocator
from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

warnings.filterwarnings("ignore")

# ── Reproducibility ──────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
keras.utils.set_random_seed(SEED)

# ── Hyperparameter ───────────────────────────────────────────────
IMG_SIZE    = (224, 224)   # ukuran input MobileNetV2
BATCH_SIZE  = 32
VAL_SPLIT   = 0.2          # proporsi validasi dari folder Training
EPOCHS_HEAD = 15           # fase 1: melatih classifier head (base model dibekukan)
EPOCHS_FINE = 20           # fase 2: fine-tuning sebagian layer MobileNetV2
LR_HEAD     = 1e-3
LR_FINE     = 1e-5
FINE_TUNE_AT = 100         # layer ke-100 dst. pada MobileNetV2 dilatih ulang di fase 2

# ── Tampilan grafik ──────────────────────────────────────────────
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.titleweight": "bold",
    "axes.titlesize": 13,
    "axes.labelsize": 11,
})

print(f"TensorFlow : {tf.__version__}")
print(f"Keras      : {keras.__version__}")
gpus = tf.config.list_physical_devices("GPU")
print(f"GPU        : {[g.name for g in gpus] if gpus else 'tidak ada (CPU)'}")

## 2. Memuat Dataset

Dataset sudah diekstrak dan diletakkan satu folder dengan notebook ini:

```
posttest_4/
├── main.ipynb
├── Training/  ── glioma/ meningioma/ notumor/ pituitary/
└── Testing/   ── glioma/ meningioma/ notumor/ pituitary/
```

Nama sub-folder otomatis menjadi **label kelas**. Folder `Training` dipakai untuk melatih sekaligus validasi, sedangkan folder `Testing` **tidak disentuh sama sekali selama pelatihan** dan hanya dipakai pada evaluasi akhir.

In [ ]:
# Folder Training & Testing berada satu folder dengan notebook (posttest_4/)
ROOT      = Path.cwd()
TRAIN_DIR = ROOT / "Training"
TEST_DIR  = ROOT / "Testing"

assert TRAIN_DIR.is_dir() and TEST_DIR.is_dir(), (
    f"Folder 'Training' dan 'Testing' tidak ditemukan di {ROOT}. "
    "Pastikan notebook dijalankan dari dalam folder posttest_4.")

IMG_EXT     = {".jpg", ".jpeg", ".png"}
CLASS_NAMES = sorted(p.name for p in TRAIN_DIR.iterdir() if p.is_dir())
NUM_CLASSES = len(CLASS_NAMES)
CLASS_COLORS = dict(zip(CLASS_NAMES, sns.color_palette("deep", NUM_CLASSES)))

def list_images(folder: Path):
    return sorted(p for p in folder.iterdir() if p.suffix.lower() in IMG_EXT)

print(f"Folder kerja  : {ROOT}")
print(f"Jumlah kelas : {NUM_CLASSES} → {CLASS_NAMES}")

## 3. Contoh Data dari Setiap Kelas  

Sebelum membangun model, kita perlu **melihat langsung** seperti apa datanya. Di bawah ini ditampilkan **5 gambar acak dari setiap kelas** (satu baris = satu kelas).

In [ ]:
def show_samples(n_per_class: int = 5):
    fig, axes = plt.subplots(NUM_CLASSES, n_per_class,
                             figsize=(3 * n_per_class, 3 * NUM_CLASSES))
    axes = np.atleast_2d(axes)
    rng = random.Random(SEED)

    for r, cls in enumerate(CLASS_NAMES):
        files = rng.sample(list_images(TRAIN_DIR / cls), n_per_class)
        for c, f in enumerate(files):
            ax = axes[r, c]
            ax.imshow(Image.open(f).convert("RGB"))
            ax.set_xticks([]); ax.set_yticks([])
            for spine in ax.spines.values():
                spine.set_edgecolor(CLASS_COLORS[cls]); spine.set_linewidth(2.5)
            if c == 0:
                ax.set_ylabel(cls.upper(), fontsize=13, fontweight="bold",
                              color=CLASS_COLORS[cls])
    fig.suptitle("Contoh Citra MRI per Kelas", fontsize=16, fontweight="bold", y=1.0)
    plt.tight_layout()
    plt.show()

show_samples(n_per_class=5)

## 4. Distribusi Jumlah Gambar per Kelas 

Jumlah data per kelas perlu diperiksa untuk mengetahui apakah dataset **seimbang (balanced)** atau tidak. Dataset yang sangat timpang membuat model cenderung bias ke kelas mayoritas, sehingga akurasi saja bisa menyesatkan dan kita perlu memakai *class weight* atau metrik per kelas.

Grafik batang berikut membandingkan jumlah gambar pada data *Training* dan *Testing* untuk setiap kelas.

In [ ]:
def count_images(split_dir: Path) -> dict:
    return {c: len(list_images(split_dir / c)) for c in CLASS_NAMES}

df_count = pd.DataFrame({"Training": count_images(TRAIN_DIR),
                         "Testing":  count_images(TEST_DIR)})
df_count["Total"] = df_count.sum(axis=1)
display(df_count)
print(f"Total gambar : {df_count['Total'].sum():,}")
ratio = df_count["Training"].max() / df_count["Training"].min()
print(f"Rasio kelas terbanyak : terkecil (Training) = {ratio:.2f}  → "
      f"{'seimbang' if ratio < 1.5 else 'tidak seimbang'}")

df_long = (df_count.drop(columns="Total").reset_index(names="Kelas")
           .melt(id_vars="Kelas", var_name="Split", value_name="Jumlah"))

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=df_long, x="Kelas", y="Jumlah", hue="Split",
            palette=["#4C72B0", "#DD8452"], ax=ax)
for container in ax.containers:
    ax.bar_label(container, padding=3, fontsize=10, fontweight="bold")
ax.set_title("Jumlah Gambar pada Setiap Kelas")
ax.set_xlabel("Kelas"); ax.set_ylabel("Jumlah Gambar")
ax.set_ylim(0, df_long["Jumlah"].max() * 1.12)
ax.legend(title="Data")
sns.despine()
plt.tight_layout()
plt.show()

## 5. Membagi Data: Train, Validation, dan Test

Data dimuat dengan `image_dataset_from_directory`, yang membaca gambar, mengubah ukurannya menjadi `224×224`, dan memberi label berdasarkan nama folder. Pembagiannya:

| Subset | Sumber | Fungsi |
|--------|--------|--------|
| **Train** | 80% folder `Training` | memperbarui bobot model |
| **Validation** | 20% folder `Training` | memantau performa tiap epoch untuk callbacks |
| **Test** | seluruh folder `Testing` | evaluasi akhir (tidak diacak agar urutan label konsisten) |

In [ ]:
common = dict(
    image_size=IMG_SIZE,
    batch_size=None,          # batching dilakukan manual agar augmentasi/cache lebih efisien
    label_mode="int",
    class_names=CLASS_NAMES,
    color_mode="rgb",         # MRI grayscale diubah ke 3 kanal sesuai input MobileNetV2
)

train_raw = keras.utils.image_dataset_from_directory(
    TRAIN_DIR, validation_split=VAL_SPLIT, subset="training",   seed=SEED, shuffle=True, **common)
val_raw = keras.utils.image_dataset_from_directory(
    TRAIN_DIR, validation_split=VAL_SPLIT, subset="validation", seed=SEED, shuffle=True, **common)
test_raw = keras.utils.image_dataset_from_directory(
    TEST_DIR, shuffle=False, **common)

print(f"\nTrain      : {len(train_raw.file_paths):>5} gambar")
print(f"Validation : {len(val_raw.file_paths):>5} gambar")
print(f"Test       : {len(test_raw.file_paths):>5} gambar")

## 6. Data Augmentation  

*Data augmentation* membuat variasi baru dari gambar yang sudah ada secara acak setiap kali gambar dimuat. Tujuan: mengurangi **overfitting** dan membuat model lebih tahan terhadap perbedaan posisi, ukuran, dan pencahayaan gambar.

Transformasi yang dipilih (disesuaikan dengan citra MRI):

| Layer | Efek | Alasan |
|-------|------|--------|
| `RandomFlip("horizontal")` | cermin kiri-kanan | otak relatif simetris; *flip* vertikal sengaja tidak dipakai karena tidak realistis |
| `RandomRotation(0.08)` | rotasi ±29° | posisi kepala pasien tidak selalu sama |
| `RandomZoom(0.1)` | zoom ±10% | variasi jarak/crop area otak |
| `RandomTranslation(0.05)` | geser ±5% | objek tidak selalu di tengah |
| `RandomContrast(0.15)` & `RandomBrightness(0.1)` | variasi kontras dan kecerahan | perbedaan mesin/pengaturan MRI |

Area kosong akibat rotasi/zoom diisi hitam (`fill_mode="constant"`), selaras dengan latar belakang citra MRI. Augmentasi **hanya diterapkan pada data train**; data validasi dan test dibiarkan asli agar evaluasi adil.

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08, fill_mode="constant"),
    layers.RandomZoom(0.10, fill_mode="constant"),
    layers.RandomTranslation(0.05, 0.05, fill_mode="constant"),
    layers.RandomContrast(0.15),
    layers.RandomBrightness(0.10, value_range=(0, 255)),
], name="data_augmentation")

Contoh hasil augmentasi untuk setiap kelas. Kolom pertama adalah gambar **asli**, kolom berikutnya adalah **4 variasi acak** dari gambar yang sama.

In [ ]:
def show_augmentation(n_aug: int = 4):
    fig, axes = plt.subplots(NUM_CLASSES, n_aug + 1,
                             figsize=(3 * (n_aug + 1), 3 * NUM_CLASSES))
    axes = np.atleast_2d(axes)
    rng = random.Random(SEED + 1)

    for r, cls in enumerate(CLASS_NAMES):
        path = rng.choice(list_images(TRAIN_DIR / cls))
        img = np.array(Image.open(path).convert("RGB").resize(IMG_SIZE), dtype="float32")
        aug = np.array(data_augmentation(np.repeat(img[None], n_aug, axis=0), training=True))
        aug = np.clip(aug, 0, 255).astype("uint8")

        tiles = [img.astype("uint8")] + list(aug)
        for c, tile in enumerate(tiles):
            ax = axes[r, c]
            ax.imshow(tile)
            ax.set_xticks([]); ax.set_yticks([])
            for spine in ax.spines.values():
                spine.set_edgecolor(CLASS_COLORS[cls]); spine.set_linewidth(2.5)
            if r == 0:
                ax.set_title("Asli" if c == 0 else f"Augmentasi {c}", fontsize=12)
            if c == 0:
                ax.set_ylabel(cls.upper(), fontsize=13, fontweight="bold",
                              color=CLASS_COLORS[cls])
    fig.suptitle("Contoh Hasil Data Augmentation per Kelas",
                 fontsize=16, fontweight="bold", y=1.0)
    plt.tight_layout()
    plt.show()

show_augmentation(n_aug=4)

## 7. Membangun Input Pipeline (`tf.data`)
1. **`cache()`** menyimpan gambar di memori setelah pembacaan pertama sehingga epoch berikutnya tidak membaca disk lagi. Gambar disimpan sebagai `uint8` agar hemat RAM (±4× lebih kecil dibanding `float32`).
2. **`shuffle()`** mengacak urutan data tiap epoch (hanya data train).
3. **`batch()`** mengelompokkan data per `BATCH_SIZE`.
4. **`map(augment)`** menerapkan augmentasi acak (hanya data train).
5. **`prefetch()`** menyiapkan batch berikutnya selagi GPU/CPU memproses batch saat ini.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

to_uint8 = lambda x, y: (tf.cast(tf.round(x), tf.uint8), y)
to_float = lambda x, y: (tf.cast(x, tf.float32), y)
augment  = lambda x, y: (data_augmentation(x, training=True), y)

train_ds = (train_raw.map(to_uint8, AUTOTUNE).cache()
            .shuffle(2048, seed=SEED).batch(BATCH_SIZE)
            .map(to_float, AUTOTUNE).map(augment, AUTOTUNE)
            .prefetch(AUTOTUNE))

val_ds = (val_raw.map(to_uint8, AUTOTUNE).cache()
          .batch(BATCH_SIZE).map(to_float, AUTOTUNE).prefetch(AUTOTUNE))

test_ds = (test_raw.map(to_uint8, AUTOTUNE).cache()
           .batch(BATCH_SIZE).map(to_float, AUTOTUNE).prefetch(AUTOTUNE))

images, labels = next(iter(train_ds))
print(f"Bentuk 1 batch → gambar: {images.shape}, label: {labels.shape}")

## 8. Membangun Model: Transfer Learning dengan MobileNetV2  *(Kriteria 4)*

Menggunakan **MobileNetV2** yang sudah dilatih pada ImageNet (*transfer learning*). Cara ini **lebih cepat konvergen dan butuh data lebih sedikit**

**Arsitektur:**

Pada awalnya seluruh bobot MobileNetV2 **dibekukan** (`trainable=False`), sehingga hanya *classifier head* baru yang dilatih. Argumen `training=False` saat memanggil base model menjaga layer *BatchNormalization* tetap memakai statistik ImageNet.

In [ ]:
def build_model(num_classes: int, dropout: float = 0.3):
    base = keras.applications.MobileNetV2(
        input_shape=IMG_SIZE + (3,),
        include_top=False,
        weights="imagenet",
        name="mobilenetv2_base",
    )
    base.trainable = False                       # fase 1: bekukan base model

    inputs  = keras.Input(shape=IMG_SIZE + (3,), name="image")
    x = layers.Rescaling(1.0 / 127.5, offset=-1, name="preprocess")(inputs)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dropout(dropout, name="dropout_1")(x)
    x = layers.Dense(128, activation="relu", name="dense_128")(x)
    x = layers.Dropout(dropout - 0.1, name="dropout_2")(x)
    outputs = layers.Dense(num_classes, activation="softmax", name="predictions")(x)

    return keras.Model(inputs, outputs, name="MobileNetV2_BrainMRI"), base

model, base_model = build_model(NUM_CLASSES)

model.compile(
    optimizer=keras.optimizers.Adam(LR_HEAD),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"],
)
model.summary()

n_train = int(np.sum([np.prod(w.shape) for w in model.trainable_weights]))
n_total = model.count_params()
print(f"\nParameter dilatih : {n_train:,} dari {n_total:,} ({n_train / n_total:.1%})")

## 9. Callbacks 

*Callbacks* adalah fungsi yang dijalankan otomatis di akhir tiap epoch. Tiga callbacks dipakai:

| Callback | Fungsi |
|----------|--------|
| **`EarlyStopping`** | **Menghentikan pelatihan** bila `val_loss` tidak membaik selama `patience` epoch, lalu mengembalikan bobot terbaik (`restore_best_weights=True`). Ini mencegah overfitting dan membuang waktu. |
| **`ReduceLROnPlateau`** | Memperkecil *learning rate* (×0,3) saat `val_loss` stagnan, sehingga model bisa melakukan penyesuaian yang lebih halus. |
| **`ModelCheckpoint`** | Menyimpan model dengan `val_loss` terbaik ke file `.keras` selama pelatihan. |

`val_loss` dipilih sebagai metrik pantauan karena lebih sensitif terhadap overfitting dibanding akurasi.

In [ ]:
def make_callbacks(tag: str, patience: int):
    return [
        keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=patience,
            restore_best_weights=True, verbose=1),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.3, patience=max(1, patience // 2),
            min_lr=1e-7, verbose=1),
        keras.callbacks.ModelCheckpoint(
            f"best_{tag}.keras", monitor="val_loss",
            save_best_only=True, verbose=0),
    ]

## 10. Pelatihan 1: Melatih Classifier Head

Pada fase pertama, MobileNetV2 masih **beku**; hanya lapisan `Dense` baru yang dilatih dengan *learning rate* relatif besar (`1e-3`). Fase ini membuat *head* yang masih berbobot acak "menyesuaikan diri" dengan fitur dari MobileNetV2 tanpa merusak bobot pretrained.

Jumlah epoch maksimum diatur `EPOCHS_HEAD`, tetapi pelatihan akan berhenti lebih awal oleh `EarlyStopping` jika tidak ada perkembangan.

In [ ]:
history_head = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_HEAD,
    callbacks=make_callbacks("head", patience=4),
)

## 11. Pelatihan 2: Fine-Tuning

Setelah *head* stabil, **buka sebagian lapisan akhir MobileNetV2** (mulai layer ke-`FINE_TUNE_AT`) agar fitur tingkat tinggi menyesuaikan diri dengan karakteristik citra MRI otak. Lapisan awal tetap beku karena hanya berisi fitur umum.

yang perlu diperhatikan:
- **Learning rate sangat kecil** (`1e-5`) agar bobot pretrained tidak rusak.
- Layer **BatchNormalization tetap dibekukan** untuk menjaga kestabilan statistik.
- Model harus di-**compile ulang** setelah mengubah status `trainable`.
- Pelatihan dilanjutkan dari epoch terakhir fase 1 (`initial_epoch`), sehingga grafik histori menyambung.

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:FINE_TUNE_AT]:
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(LR_FINE),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"],
)

n_train = int(np.sum([np.prod(w.shape) for w in model.trainable_weights]))
print(f"Parameter dilatih saat fine-tuning : {n_train:,} dari {model.count_params():,}\n")

start_epoch  = len(history_head.epoch)          # epoch yang benar-benar dijalankan di fase 1
history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=start_epoch + EPOCHS_FINE,
    initial_epoch=start_epoch,
    callbacks=make_callbacks("fine", patience=5),
)

## 12. Grafik Histori Pelatihan 

Histori dari kedua fase digabung menjadi satu grafik **akurasi** dan **loss** untuk data train dan validasi.

In [ ]:
def merge_history(*histories):
    merged = {}
    for h in histories:
        for k, v in h.history.items():
            merged.setdefault(k, []).extend(v)
    return merged

hist       = merge_history(history_head, history_fine)
epochs     = np.arange(1, len(hist["loss"]) + 1)
split_at   = len(history_head.epoch)
best_epoch = int(np.argmin(hist["val_loss"])) + 1

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for ax, metric, title in zip(axes, ["accuracy", "loss"],
                             ["Akurasi Model", "Loss Model"]):
    ax.plot(epochs, hist[metric],          "-o", ms=4, lw=2, label="Train",      color="#4C72B0")
    ax.plot(epochs, hist[f"val_{metric}"], "-o", ms=4, lw=2, label="Validation", color="#DD8452")
    ax.axvline(split_at + 0.5, ls="--", color="gray", alpha=.8)
    ax.text(split_at + 0.6, 0.03, "mulai fine-tuning", color="gray", fontsize=9,
            transform=ax.get_xaxis_transform())
    ax.scatter(best_epoch, hist[f"val_{metric}"][best_epoch - 1], marker="*", s=260,
               color="gold", edgecolor="black", zorder=5, label=f"Epoch terbaik ({best_epoch})")
    ax.set_title(title); ax.set_xlabel("Epoch"); ax.set_ylabel(metric.capitalize())
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.legend(loc="best")
sns.despine()
plt.tight_layout()
plt.show()

print(f"Total epoch dijalankan : {len(epochs)}  (fase 1: {split_at}, fase 2: {len(epochs) - split_at})")
print(f"Epoch terbaik          : {best_epoch}")
print(f"  val_loss     = {hist['val_loss'][best_epoch - 1]:.4f}")
print(f"  val_accuracy = {hist['val_accuracy'][best_epoch - 1]:.4f}")

## 13. Evaluasi Model pada Data Test 

Model dievaluasi pada data **Testing** yang belum pernah dilihat selama pelatihan. Tahap ini terdiri dari:
1. menghitung akurasi dan loss pada data test, serta menghasilkan prediksi,
2. **confusion matrix** untuk melihat kesalahan antar-kelas,
3. **classification report** (precision, recall, F1-score),
4. contoh visual prediksi.

### 13.1 Prediksi dan Skor Keseluruhan

In [ ]:
test_loss, test_acc = model.evaluate(test_ds, verbose=0)

y_prob = model.predict(test_ds, verbose=0)               # probabilitas tiap kelas
y_pred = y_prob.argmax(axis=1)                           # kelas dengan probabilitas tertinggi
y_true = np.concatenate([y.numpy() for _, y in test_ds]) # label asli (urutan tidak diacak)

print(f"Test Loss     : {test_loss:.4f}")
print(f"Test Accuracy : {test_acc:.4f}  ({test_acc:.2%})")
print(f"Benar / total : {(y_true == y_pred).sum()} / {len(y_true)}")

### 13.2 Confusion Matrix

Confusion matrix menunjukkan **berapa gambar dari tiap kelas asli (baris) yang diprediksi sebagai kelas tertentu (kolom)**.
- **Diagonal** adalah prediksi yang benar, semakin gelap/besar semakin baik.
- **Di luar diagonal** adalah kesalahan, dan dari sini terlihat pasangan kelas yang sering tertukar.

Panel kiri menampilkan jumlah gambar, sedangkan panel kanan menampilkan persentase per baris (sama dengan *recall* tiap kelas), yang lebih adil dibaca bila jumlah data antar-kelas berbeda.

In [ ]:
cm      = confusion_matrix(y_true, y_pred)
cm_norm = cm / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", linewidths=.5, cbar=False,
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            annot_kws={"size": 13, "weight": "bold"}, ax=axes[0])
axes[0].set_title("Confusion Matrix (Jumlah)")

sns.heatmap(cm_norm, annot=True, fmt=".1%", cmap="Greens", linewidths=.5, cbar=False,
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            annot_kws={"size": 13, "weight": "bold"}, ax=axes[1])
axes[1].set_title("Confusion Matrix (Persentase per Kelas Asli)")

for ax in axes:
    ax.set_xlabel("Kelas Prediksi"); ax.set_ylabel("Kelas Asli")
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

### 13.3 Classification Report

Definisi metrik untuk setiap kelas:
- **Precision**: dari semua gambar yang *diprediksi* sebagai kelas positif, berapa persen yang benar-benar positif. Precision rendah berarti banyak *false positive*.
- **Recall**: dari semua gambar yang *sebenarnya* kelas positif, berapa persen yang berhasil ditemukan. Recall rendah berarti banyak *false negative*. Pada konteks medis, ini biasanya yang paling kritis karena kasus tumor yang terlewat berisiko.
- **F1-score**: rata-rata harmonik precision dan recall.
- **Support**: jumlah data asli kelas tersebut pada data test.

In [ ]:
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4))

### 13.4 Contoh Hasil Prediksi

Sepuluh gambar test dipilih secara acak. Judul **hijau** berarti prediksi benar, **merah** berarti salah. Angka dalam kurung adalah tingkat keyakinan (probabilitas) model.

In [ ]:
rng = np.random.default_rng(SEED)
idx = rng.choice(len(y_true), size=10, replace=False)

fig, axes = plt.subplots(2, 5, figsize=(16, 7.5))
for ax, i in zip(axes.ravel(), idx):
    ax.imshow(Image.open(test_raw.file_paths[i]).convert("RGB"))
    ok = y_true[i] == y_pred[i]
    ax.set_title(f"Asli: {CLASS_NAMES[y_true[i]]}\n"
                 f"Prediksi: {CLASS_NAMES[y_pred[i]]} ({y_prob[i].max():.0%})",
                 color="seagreen" if ok else "crimson", fontsize=10.5)
    ax.axis("off")
fig.suptitle("Contoh Prediksi pada Data Test", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

## 14. Menyimpan Model & Ringkasan Hasil

Model akhir disimpan dalam format `.keras` agar dapat dimuat kembali untuk inferensi tanpa perlu melatih ulang (`keras.models.load_model(...)`). Sel di bawah juga merangkum hasil secara otomatis.

In [ ]:
MODEL_PATH = "model_mobilenetv2_brain_mri.keras"
model.save(MODEL_PATH)
print(f"Model disimpan ke: {MODEL_PATH}\n")

report = classification_report(y_true, y_pred, target_names=CLASS_NAMES, output_dict=True)
best_cls  = max(CLASS_NAMES, key=lambda c: report[c]["f1-score"])
worst_cls = min(CLASS_NAMES, key=lambda c: report[c]["f1-score"])

off = cm.copy(); np.fill_diagonal(off, 0)
i, j = np.unravel_index(off.argmax(), off.shape)

print("═" * 58)
print(f" Akurasi test           : {test_acc:.2%}")
print(f" Macro F1-score         : {report['macro avg']['f1-score']:.4f}")
print(f" Kelas terbaik (F1)     : {best_cls} ({report[best_cls]['f1-score']:.4f})")
print(f" Kelas terlemah (F1)    : {worst_cls} ({report[worst_cls]['f1-score']:.4f})")
if off.max() > 0:
    print(f" Kesalahan terbanyak    : '{CLASS_NAMES[i]}' diprediksi '{CLASS_NAMES[j]}' ({off.max()} gambar)")
print("═" * 58)